In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers

import numpy as np
import tensorflow as tf
import gymnasium as gym
import scipy.signal

In [2]:
def discounted_cumulative_sums(x, discount):
    return scipy.signal.lfilter([1], [1, float(-discount)], x[::-1], axis=0)[::-1]


class Buffer:
    def __init__(self, observation_dimensions, size, gamma=0.99, lam=0.95):
        self.observation_buffer = np.zeros(
            (size, observation_dimensions), dtype=np.float32
        )
        self.action_buffer = np.zeros(size, dtype=np.int32)
        self.advantage_buffer = np.zeros(size, dtype=np.float32)
        self.reward_buffer = np.zeros(size, dtype=np.float32)
        self.return_buffer = np.zeros(size, dtype=np.float32)
        self.value_buffer = np.zeros(size, dtype=np.float32)
        self.logprobability_buffer = np.zeros(size, dtype=np.float32)
        self.gamma, self.lam = gamma, lam
        self.pointer, self.trajectory_start_index = 0, 0

    def store(self, observation, action, reward, value, logprobability):
        self.observation_buffer[self.pointer] = observation
        self.action_buffer[self.pointer] = action
        self.reward_buffer[self.pointer] = reward
        self.value_buffer[self.pointer] = value
        self.logprobability_buffer[self.pointer] = logprobability
        self.pointer += 1

    def finish_trajectory(self, last_value=0):
        path_slice = slice(self.trajectory_start_index, self.pointer)
        rewards = np.append(self.reward_buffer[path_slice], last_value)
        values = np.append(self.value_buffer[path_slice], last_value)

        deltas = rewards[:-1] + self.gamma * values[1:] - values[:-1]

        self.advantage_buffer[path_slice] = discounted_cumulative_sums(
            deltas, self.gamma * self.lam
        )
        self.return_buffer[path_slice] = discounted_cumulative_sums(
            rewards, self.gamma
        )[:-1]

        self.trajectory_start_index = self.pointer

    def get(self):
        self.pointer, self.trajectory_start_index = 0, 0
        advantage_mean, advantage_std = (
            np.mean(self.advantage_buffer),
            np.std(self.advantage_buffer),
        )
        self.advantage_buffer = (self.advantage_buffer - advantage_mean) / advantage_std
        return (
            self.observation_buffer,
            self.action_buffer,
            self.advantage_buffer,
            self.return_buffer,
            self.logprobability_buffer,
        )


def mlp(x, sizes, activation=keras.activations.tanh, output_activation=None):
    for size in sizes[:-1]:
        x = layers.Dense(units=size, activation=activation)(x)
    return layers.Dense(
          units=int(sizes[-1]),
          activation=output_activation
      )(x)


def logprobabilities(logits, a):
    logprobabilities_all = keras.ops.log_softmax(logits)
    logprobability = keras.ops.sum(
        keras.ops.one_hot(a, num_actions) * logprobabilities_all, axis=1
    )
    return logprobability


seed_generator = keras.random.SeedGenerator(1337)

@tf.function
def sample_action(observation):
    logits = actor(observation)
    action = keras.ops.squeeze(
        keras.random.categorical(logits, 1, seed=seed_generator), axis=1
    )
    return logits, action


@tf.function
def train_policy(
    observation_buffer, action_buffer, logprobability_buffer, advantage_buffer
):
    with tf.GradientTape() as tape:
        ratio = keras.ops.exp(
            logprobabilities(actor(observation_buffer), action_buffer)
            - logprobability_buffer
        )
        min_advantage = keras.ops.where(
            advantage_buffer > 0,
            (1 + clip_ratio) * advantage_buffer,
            (1 - clip_ratio) * advantage_buffer,
        )

        policy_loss = -keras.ops.mean(
            keras.ops.minimum(ratio * advantage_buffer, min_advantage)
        )
    policy_grads = tape.gradient(policy_loss, actor.trainable_variables)
    policy_optimizer.apply_gradients(zip(policy_grads, actor.trainable_variables))

    kl = keras.ops.mean(
        logprobability_buffer
        - logprobabilities(actor(observation_buffer), action_buffer)
    )
    kl = keras.ops.sum(kl)
    return kl


@tf.function
def train_value_function(observation_buffer, return_buffer):
    with tf.GradientTape() as tape:
        value_loss = keras.ops.mean((return_buffer - critic(observation_buffer)) ** 2)
    value_grads = tape.gradient(value_loss, critic.trainable_variables)
    value_optimizer.apply_gradients(zip(value_grads, critic.trainable_variables))

In [3]:
steps_per_epoch = 4000
epochs = 20
gamma = 0.99
clip_ratio = 0.2
policy_learning_rate = 3e-4
value_function_learning_rate = 1e-3
train_policy_iterations = 80
train_value_iterations = 80
lam = 0.97
target_kl = 0.01
hidden_sizes = (64, 64)

render = False

In [4]:
env = gym.make("CartPole-v1")
observation_dimensions = env.observation_space.shape[0]
num_actions = env.action_space.n

buffer = Buffer(observation_dimensions, steps_per_epoch)

observation_input = keras.Input(shape=(observation_dimensions,), dtype="float32")
logits = mlp(observation_input, list(hidden_sizes) + [num_actions])
actor = keras.Model(inputs=observation_input, outputs=logits)
value = keras.ops.squeeze(mlp(observation_input, list(hidden_sizes) + [1]), axis=1)
critic = keras.Model(inputs=observation_input, outputs=value)

policy_optimizer = keras.optimizers.Adam(learning_rate=policy_learning_rate)
value_optimizer = keras.optimizers.Adam(learning_rate=value_function_learning_rate)

observation, _ = env.reset()
episode_return, episode_length = 0, 0

In [5]:
for epoch in range(epochs):
    sum_return = 0
    sum_length = 0
    num_episodes = 0

    for t in range(steps_per_epoch):
        if render:
            env.render()

        observation = observation.reshape(1, -1)
        logits, action = sample_action(observation)
        observation_new, reward, done, _, _ = env.step(action[0].numpy())
        episode_return += reward
        episode_length += 1

        value_t = critic(observation)
        logprobability_t = logprobabilities(logits, action)

        buffer.store(observation, action, reward, value_t, logprobability_t)

        observation = observation_new

        terminal = done
        if terminal or (t == steps_per_epoch - 1):
            last_value = 0 if done else critic(observation.reshape(1, -1))
            buffer.finish_trajectory(last_value)
            sum_return += episode_return
            sum_length += episode_length
            num_episodes += 1
            observation, _ = env.reset()
            episode_return, episode_length = 0, 0

    (
        observation_buffer,
        action_buffer,
        advantage_buffer,
        return_buffer,
        logprobability_buffer,
    ) = buffer.get()

    for _ in range(train_policy_iterations):
        kl = train_policy(
            observation_buffer, action_buffer, logprobability_buffer, advantage_buffer
        )
        if kl > 1.5 * target_kl:
            break

    for _ in range(train_value_iterations):
        train_value_function(observation_buffer, return_buffer)

    print(
        f" Epoch: {epoch + 1}. Mean Return: {sum_return / num_episodes}. Mean Length: {sum_length / num_episodes}"
    )

 Epoch: 1. Mean Return: 21.62162162162162. Mean Length: 21.62162162162162
 Epoch: 2. Mean Return: 25.31645569620253. Mean Length: 25.31645569620253
 Epoch: 3. Mean Return: 35.714285714285715. Mean Length: 35.714285714285715
 Epoch: 4. Mean Return: 48.19277108433735. Mean Length: 48.19277108433735
 Epoch: 5. Mean Return: 75.47169811320755. Mean Length: 75.47169811320755
 Epoch: 6. Mean Return: 114.28571428571429. Mean Length: 114.28571428571429
 Epoch: 7. Mean Return: 166.66666666666666. Mean Length: 166.66666666666666
 Epoch: 8. Mean Return: 266.6666666666667. Mean Length: 266.6666666666667
 Epoch: 9. Mean Return: 285.7142857142857. Mean Length: 285.7142857142857
 Epoch: 10. Mean Return: 444.44444444444446. Mean Length: 444.44444444444446
 Epoch: 11. Mean Return: 571.4285714285714. Mean Length: 571.4285714285714
 Epoch: 12. Mean Return: 1000.0. Mean Length: 1000.0
 Epoch: 13. Mean Return: 800.0. Mean Length: 800.0
 Epoch: 14. Mean Return: 2000.0. Mean Length: 2000.0
 Epoch: 15. Mean Re

In [6]:
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video
import glob

print("Recording the trained AI...")

# 1. Use 'rgb_array' instead of 'human'
base_env = gym.make("CartPole-v1", render_mode="rgb_array")

# 2. Wrap the environment to automatically record MP4 videos
test_env = RecordVideo(base_env, video_folder='./video', name_prefix='cartpole-agent', disable_logger=True)

observation, _ = test_env.reset()
done = False
steps_survived = 0

while not done:
    # Get the action from your trained actor model
    logits = actor(observation.reshape(1, -1))
    action = np.argmax(logits.numpy()[0])

    observation, reward, terminated, truncated, _ = test_env.step(action)
    done = terminated or truncated
    steps_survived += 1

print(f"Episode finished! The AI balanced the pole for {steps_survived} steps.")
test_env.close()

# 3. Find the saved video file and display it in the notebook
video_file = glob.glob('./video/cartpole-agent*.mp4')[0]
display(Video(video_file, embed=True, html_attributes="controls autoplay loop"))

Recording the trained AI...


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode finished! The AI balanced the pole for 500 steps.
